# Developer Insights Bible

## End-to-End Architecture, Schemas, Connectors, and Delivery Blueprint

Repository: `/Users/sebbo/Desktop/Replit revival`  
Date: 2026-03-06

This document is the canonical developer reference for insights, narratives, commentary, quotes, and chart text flows.

---

## 1) Global Map of Insights / Quotes / Commentary Logic

### 1.1 System View
The app uses three parallel text pipelines:

1. Workbook-curated text
- Source tabs include `Overview_Insights`, `Overview_Charts`, `Overview_Macro`, `Company_insights_text`, `Company_Segments_insights_text`, `Overview_Auto_Insights`, `Transcripts`.
- Rendered in `app/pages/00_Overview.py` and `app/pages/01_Earnings.py`.

2. Transcript intelligence pipeline
- Local transcript files (`earningscall_transcripts/<Company>/<Year>/Qx.txt`) are indexed and transformed into CSV/SQLite artifacts.
- Scripts: `rebuild_transcript_index.py`, `extract_transcript_topics.py`, `extract_transcript_highlights_from_sheet.py`, `build_intelligence_db.py`, `sync_gsheet_to_sql.py`, `generate_insights.py`, `sync_all_intelligence.py`.

3. Runtime AI commentary
- `app/pages/04_Genie.py` + `app/utils/openai_service.py` + `app/utils/genie_ai.py`.
- Uses workbook/CSV/SQLite context and generates dynamic textual output.

### 1.2 Mermaid: Workbook to UI
```mermaid
flowchart LR
    GS["Google Sheet (43 tabs)"] --> WS["app/utils/workbook_source.py"]
    WS --> DP["app/data_processor.py"]
    WS --> WM["app/utils/workbook_market_data.py"]
    WM --> SP["app/stock_processor_fix.py"]
    DP --> OVR["app/pages/00_Overview.py"]
    DP --> ERN["app/pages/01_Earnings.py"]
    DP --> GEN["app/pages/04_Genie.py"]
    SP --> ERN
```

### 1.3 Mermaid: Transcript Pipeline
```mermaid
flowchart TD
    TXT["earningscall_transcripts/*.txt"] --> IDX["scripts/rebuild_transcript_index.py"]
    IDX --> TOP["scripts/extract_transcript_topics.py"]
    IDX --> HIL["scripts/extract_transcript_highlights_from_sheet.py"]
    TOP --> CSV1["transcript_topics.csv / transcript_kpis.csv / topic_metrics.csv"]
    HIL --> CSV2["transcript_highlights.csv / overview_iconic_quotes.csv"]
    CSV1 --> DBB["scripts/build_intelligence_db.py"]
    CSV2 --> DBB
    DBB --> SQL["earningscall_intelligence.db"]
    SQL --> GEN["app/pages/04_Genie.py"]
    CSV2 --> OVR["app/pages/00_Overview.py"]
    CSV2 --> ERN["app/pages/01_Earnings.py"]
```

### 1.4 Plotly Assets
- Pipeline dependency graph: `/tmp/dev_manual_assets/pipeline_dependency_graph.png`
- 43-sheet usage heatmap: `/tmp/dev_manual_assets/sheet_usage_heatmap.png`

### 1.5 Major Domain Files and Roles
| File | Role |
|---|---|
| `app/utils/workbook_source.py` | Google workbook source resolver + coverage guardrails |
| `app/utils/workbook_market_data.py` | Daily/Minute/Holders normalization and dedup |
| `app/data_processor.py` | Primary financial/text sheet loader |
| `app/stock_processor_fix.py` | Stock historical processor using combined market sheets |
| `app/pages/00_Overview.py` | Overview insights, chart comments, quote cards |
| `app/pages/01_Earnings.py` | Earnings company/segment insights + transcript quotes |
| `app/pages/04_Genie.py` | Conversational insights path |
| `scripts/*transcript*` | Transcript extraction/index/highlights/topic generation |
| `scripts/build_intelligence_db.py` | CSV → SQLite materialization |
| `scripts/sync_gsheet_to_sql.py` | Workbook metrics sync to SQLite |
| `scripts/generate_insights.py` | Auto-generated insight text rows |

---

## 2) Data Sources and Schemas (Text + Financial)

### 2.1 Primary Source of Truth
`app/utils/workbook_source.py` resolves workbook from Google export endpoint:
- `https://docs.google.com/spreadsheets/d/<sheet_id>/export?format=xlsx`
- Default workbook ID: `10pOfzRRd0Mhbb_jq_fQRCqNr7N_R2KUpBBNsUuo5sxs`
- Expected tabs: at least 43
- Required tabs: `Company_metrics_earnings_values`, `Daily`, `Minute`, `Holders`

### 2.2 Full 43-Sheet List (Current Runtime Snapshot)
1. Daily
2. Minute
3. Holders
4. USD Inflation
5. Nasdaq Composite Est. (FRED)
6. Country_Totals_vs_GDP
7. Country_Totals_vs_GDP_ RAW
8. Country_Advertising_Data_FullVi
9. Country_avg_timespent_intrnt24
10. Global_Adv_Aggregates
11. Global Advertising (GroupM)
12.  (GroupM) Granular 
13. M2_values
14. Company_metrics_earnings_values
15. Company_Employees
16. Company_Segments_insights_text
17. Company_insights_text
18. Company_advertising_revenue
19. Company_subscribers_values
20. Hardware_Smartphone_Shipments
21. Macro_Wealth_by_Generation
22. Company_revenue_by_region
23. Company_minute&dollar_earned
24. Company_yearly_segments_values
25. Company_Quarterly_segments_valu
26. Alphabet Quarterly Segments
27. Apple Quarterly Segments
28. Amazon Quarterly Segments 
29. Meta Quarterly Segments
30. Comcast Quarterly Segments Gran
31. Disney Quarterly Segments
32. Microsoft Quarterly Segments
33. Netflix Quarterly Segments
34. Paramount Quarterly Segments
35. Roku Quarterly Segments
36. Spotify Quarterly Segments
37. Warner Bros Quarterly Segments
38. Overview_Macro
39. Overview_Insights
40. Overview_Charts
41. Transcripts
42. Overview_Auto_Insights
43. Macro_KPIs

### 2.3 Required New Sheet Schemas
#### Daily
| Column | Type | Example |
|---|---|---|
| date | date/datetime | 2026-03-05 |
| price | float | 182.41 |
| open | float | 178.20 |
| high | float | 184.00 |
| low | float | 177.95 |
| vol. | float/int | 12400000 |
| change % | float/string | +2.36% |
| market cap. | float/string | 2.85T |
| currency | string | USD |
| asset | string | AAPL |
| outstanding shares | float/int | 15400000000 |
| tag | string | equity |

#### Minute
| Column | Type | Example |
|---|---|---|
| date | datetime | 2026-03-05 15:59:00 |
| close | float | 182.41 |
| open | float | 182.10 |
| high | float | 182.55 |
| low | float | 181.90 |
| volume | float/int | 210530 |
| change % | float/string | +0.17% |
| market cap | float/string | 2.85T |
| currency | string | USD |
| asset | string | AAPL |

#### Holders
| Column | Type | Example |
|---|---|---|
| date_fetched | datetime | 2026-03-05 00:00:00 |
| company | string | Apple |
| ticker | string | AAPL |
| holder_name | string | Vanguard Group |
| shares | float/int | 1300000000 |
| value_usd | float | 236000000000 |
| pct_out | float | 8.43 |
| holder_type | string | Institution |

### 2.4 CSV Sources (Operational)
| CSV | Purpose |
|---|---|
| `transcript_index.csv` | transcript inventory |
| `transcript_topics.csv` | extracted topic rows |
| `transcript_kpis.csv` | extracted KPI rows |
| `transcript_highlights.csv` | highlights and quote rows |
| `overview_iconic_quotes.csv` | curated iconic CEO/CFO quotes |
| `topic_metrics.csv` | topic-level aggregate metrics |
| `generated_insights_latest.csv` | generated insight rows for fallback/rendering |

### 2.5 SQLite Tables
| Table | Purpose |
|---|---|
| `transcripts` | transcript master rows |
| `transcript_topics` | topic extraction rows |
| `transcript_kpis` | KPI extraction rows |
| `transcript_highlights` | quote/highlight rows |
| `company_metrics` | company financial metrics + enrichments |

---

## 3) Full Text Pipeline(s)

### 3.1 Extraction
| Function | File | Reads From |
|---|---|---|
| `_load_overview_insights_sheet` | `app/pages/00_Overview.py` | `Overview_Insights` |
| `_load_overview_charts_sheet` | `app/pages/00_Overview.py` | `Overview_Charts` |
| `_load_overview_iconic_quotes_sheet` | `app/pages/00_Overview.py` | quote sheet aliases |
| `load_company_segment_insights` | `app/pages/01_Earnings.py` | `Company_Segments_insights_text` |
| `load_company_insights_text` | `app/pages/01_Earnings.py` | `Company_insights_text` |
| `_load_transcript_highlights_csv` | `app/pages/01_Earnings.py` | `transcript_highlights.csv` |

### 3.2 Transformation
| Function | Behavior |
|---|---|
| `_rename_overview_columns` | maps alias headers to canonical schema |
| `_normalize_iconic_quotes_df` | year/quarter coercion + role normalization |
| `_parse_quarter_int` | quarter parsing from text/number formats |
| `_normalize_stock_sheet` | stock tab column normalization |
| `load_combined_stock_market_data` | baseline+Daily+Minute merge + dedup |
| `load_holders_sheet` | holders map + parse + dedup |

### 3.3 Rendering
| Renderer | Output Pattern |
|---|---|
| `_render_excel_overview_insights` | category cards with title/stat/comment/logos |
| `_render_iconic_quote_section` | quote grid cards |
| earnings segment insight block | colored segment insight cards + hover map |
| `render_transcript_highlights` | CEO/CFO quote blocks in markdown |
| Genie response path | conversational generated narratives |

### 3.4 Example Flow A
`Overview_Insights` row → `_load_overview_insights_sheet` normalize/filter → `_render_excel_overview_insights` category grouping → HTML cards in Overview page.

### 3.5 Example Flow B
`transcript_highlights.csv` → `_load_transcript_highlights_csv` normalize company/year/quarter/role bucket → `render_transcript_highlights` role-sorted quotes.

### 3.6 Example Flow C
`Daily` + `Minute` + baseline stock rows → `load_combined_stock_market_data` dedup by `(date, asset, tag)` using source priority → earnings stock charts/metrics.

---

## 4) Executive Quotes (CEO/CFO/Management)

### 4.1 Quote Sources and Fallbacks
1. Preferred: quote tab in workbook (`Overview_Iconic_Quotes` or aliases)
2. Fallback: `earningscall_transcripts/overview_iconic_quotes.csv`
3. Fallback: `earningscall_transcripts/transcript_highlights.csv`
4. Earnings page direct quote source: `transcript_highlights.csv`

### 4.2 Quote Record Shape
| Field | Meaning |
|---|---|
| year | quote year |
| quarter | quote quarter |
| company | normalized company |
| speaker | executive speaker |
| role_bucket | CEO/CFO/other normalized role |
| quote | quote text |
| score | rank/relevance |

### 4.3 Role Handling Logic
- CEO detection: contains `ceo` or `chief executive`
- CFO detection: contains `cfo` or `chief financial`
- Else uppercase fallback

### 4.4 Matching Logic
- Company normalization first
- Year exact match
- Quarter exact if selected, otherwise latest available
- Role-specific display in earnings (`CEO` then `CFO`)

### 4.5 Prepared Remarks vs Q&A
Current model does not strongly type this in UI; if upstream rows include hinting fields they can be filtered, but main rendering path is role + period driven.

---

## 5) Insights Categories / Grouping Logics

### 5.1 Grouping Rules
| Context | Group Key |
|---|---|
| Overview curated insights | `category` field |
| Overview auto insights | `category` + `priority` + `sort_order` |
| Earnings company insights | `category` field |
| Earnings segment insights | `segment` field |
| Transcript highlights | `role_bucket` (`CEO`/`CFO`) |

### 5.2 Ordering Rules
- Overview insights: sort by `sort_order`, fallback from `insight_id` numeric extraction.
- Overview quotes: sort by `score desc`, then company/speaker.
- Earnings highlights: sort by `score desc`, cap per role.
- Market merge insights: dedup precedence `Minute > Daily > Stocks & Crypto`.

### 5.3 Concrete Placement Examples
1. `Overview_Insights.category = "Macro"` row appears in Macro category card.
2. `Company_insights_text` row with `quarter=Q4` is selected over annual row when Q4 selected.
3. `transcript_highlights.role_bucket = CEO` rows render in CEO quote block first.
4. `generated_insights_latest.csv` `priority=1` row precedes `priority=2`.

---

## 6) Chart Commentary and Contextual Logic

### 6.1 Comment Types
- Pre-chart narrative (`pre_comment`)
- Post-chart takeaway (`post_comment`)
- Insight card body comments (`comment`)
- Quote cards and transcript highlight quote blocks
- Macro commentary (`macro_comment`)

### 6.2 Lookup Keys
- `company`
- `year`
- `quarter`
- `chart_key`
- `category`
- `segment`

### 6.3 Fallback and Precedence
1. Exact period + key
2. Year-level fallback
3. Category/general fallback
4. CSV fallback (quotes, generated insights)
5. Empty state message in UI

### 6.4 Missing Data Handling
- Missing quarter: quarterly match attempted, otherwise annual/blank-quarter rows used.
- Missing chart key: generic category section remains renderable.
- Multiple matches: deterministic sort (order/score) selects top rows.

---

## 7) Business Rules and Special Cases

### 7.1 Hard Guardrails
- `_has_core_financial_coverage()` requires usable metrics backbone.
- Min sheet topology and required tabs enforced.
- Workbook rejected when requirements fail.

### 7.2 Market Merge Rules
- Normalize prices and timestamps from multiple tabs.
- Dedup key: `date + asset + tag`.
- Source priority: `Stocks & Crypto` (1), `Daily` (2), `Minute` (3), keep highest.

### 7.3 Alias / Canonicalization Rules
Multiple alias maps normalize company/tickers across stock feeds and text rows.

### 7.4 Deprecated Secondary Feed
Legacy secondary live CSV connector is deprecated and intentionally disabled in `app/utils/live_stock_feed.py`.

---

## 8) Caching and Freshness of Narratives

### 8.1 Cache Layers
| Layer | Mechanism |
|---|---|
| Workbook download cache | temp `.xlsx` cache file keyed by sheet ID |
| Sheet read cache | `@lru_cache` helpers |
| Streamlit data cache | `@st.cache_data` across pages and utilities |
| SQLite ETL cache | periodic script-run materialization |

### 8.2 Known Cached Functions (Examples)
- `app/utils/workbook_market_data.py`: `_read_sheet_cached`
- `app/stock_processor_fix.py`: `_load_stock_sheet`
- `app/data_processor.py`: `_read_excel_sheet`
- `app/pages/00_Overview.py`: many cached loaders (`ttl=3600` in many cases)
- `app/pages/01_Earnings.py`: cached stock/insight/transcript loaders

### 8.3 Staleness Scenarios
1. Workbook updated online but cached local export reused.
2. ETL script fails partially; stale SQLite persists.
3. Sheet schema drift causes empty dataframes and silent content gaps.
4. Parallel output paths disagree on latest narrative.

---

## 9) Styling, Theming, and Layout Logic for Text

### 9.1 Overview Text Classes
Examples from `app/pages/00_Overview.py`:
- `ov-insight-category-card`, `ov-insight-category-title`
- `ov-insight-item`, `ov-insight-head`, `ov-insight-body`, `ov-insight-stat`
- `ov-quote-grid`, `ov-quote-card`, `ov-quote-meta`, `ov-quote-body`
- `ov-chart-comment`, `ov-chart-comment-post`

### 9.2 Earnings Text Classes
Examples from `app/pages/01_Earnings.py`:
- `company-insight-card`, `company-insight-title`, `company-insight-list`
- `segment-insight-card`, `segment-insight-title`, `segment-insight-list`
- `insights-carousel`, `insights-nav`, `insight-card`
- KPI text hierarchy classes (`kpi-label`, `kpi-value`, `kpi-yoy`)

### 9.3 Theme and Responsive Behavior
- Uses Streamlit theme plus injected CSS.
- Light/dark adaptations present for major cards.
- Insight cards/carousels collapse and stack on narrow layouts.

---

## 10) File-by-File Domain Index (Project-Owned Files)

Scope: project-owned files excluding `.venv`.

### 10.1 Directly Participating in Insights/Quotes/Commentary
- `app/Welcome.py`
- `app/handle_segments.py`
- `app/pages/00_Overview.py`
- `app/pages/01_Earnings.py`
- `app/pages/02_Stocks.py` (minor textual overlays)
- `app/pages/03_Editorial.py` (minor)
- `app/pages/04_Genie.py`
- `app/stock_processor_fix.py`
- `app/utils/genie_ai.py`
- `app/utils/insights.py`
- `app/utils/insights_loader.py`
- `app/utils/insights_loader_fixed.py`
- `app/utils/live_stock_feed.py` (legacy)
- `app/utils/macro_trends.py`
- `app/utils/openai_service.py`
- `app/utils/sql_assistant_sidebar.py` (indirect)
- `app/utils/styles.py` (UI text presentation)
- `app/utils/theme.py` (visual hierarchy)
- `app/utils/transcript_startup_sync.py` (pipeline freshness)
- `db_schema.sql`
- `earningscall_transcripts/README.md`
- `scripts/build_intelligence_db.py`
- `scripts/extract_kpi_values.py`
- `scripts/extract_transcript_highlights_from_sheet.py`
- `scripts/extract_transcript_topics.py`
- `scripts/generate_diagnostic_report.py`
- `scripts/generate_insights.py`
- `scripts/intelligence_db_schema.py`
- `scripts/rebuild_transcript_index.py`
- `scripts/sync_all_intelligence.py`
- `scripts/sync_gsheet_to_sql.py`
- `scripts/sync_iconic_quotes_to_gsheet.py`
- `scripts/upload_local_transcripts_to_gsheet.py`

### 10.2 Indirect or Non-Domain-Primary Files
All other project-owned files in the repository either:
- provide app framework/scaffolding,
- provide generic auth/api/ui helpers,
- or are disabled/legacy mirrors (`pages/` root duplicates).

---

## 11) Cross-Cutting Concerns and Hidden Logic

### 11.1 Environment Variables Affecting Pipeline
| Variable | Effect |
|---|---|
| `FINANCIAL_DATA_SOURCE` | source mode preference (Google enforced behavior) |
| `FINANCIAL_DATA_GSHEET_URL` / `FINANCIAL_DATA_GSHEET_ID` | workbook target |
| `FINANCIAL_DATA_GSHEET_REFRESH_SECONDS` | workbook refresh interval |
| `OPENAI_API_KEY` | runtime AI response path |
| `DATABASE_URL` + PG vars | DB-backed utility paths |
| `AUTO_SYNC_TRANSCRIPTS_ON_STARTUP` | startup pipeline behavior |

### 11.2 Duplicated/Inconsistent Logic
1. `insights_loader.py` and `insights_loader_fixed.py` duplicate behavior.
2. Multiple quarter parsing helpers across files.
3. Multiple number/percent formatting implementations.
4. Workbook-curated vs generated vs AI runtime text can diverge.

### 11.3 Hidden Fallback Channels
- Quote sources can silently fall back from workbook tab to CSV.
- Generated insights can fall back to local CSV when tab absent.
- Deprecated live feed functions still callable but disabled.

---

## 12) Gaps, Risks, TODOs, and Migration Blueprint

### 12.1 Top Risks
| Risk | Impact | Priority |
|---|---|---|
| stale cache / stale SQLite | old narratives shown | P0 |
| sheet schema drift | silent text gaps | P0 |
| duplicate logic paths | inconsistent outputs | P1 |
| legacy schema references | runtime branch failures | P1 |
| silent startup script failures | false app health | P0 |

### 12.2 Google Sheet Migration Checklist (Actionable)
1. Keep Google as default source (`workbook_source.py`).
2. Keep `_has_core_financial_coverage()` intact.
3. Keep export endpoint `/export?format=xlsx`.
4. Keep required tabs (`Daily`, `Minute`, `Holders`) in validation.
5. Keep live stock feed connector disabled/deprecated.
6. Ensure earnings stock reads use `load_combined_stock_market_data()`.
7. Ensure holders load through `load_holders_sheet()` and `data_processor.get_holders()`.
8. Ensure no local fallback to `Earnings + stocks  copy.xlsx` in runtime paths.
9. Preserve transcript sync scripts unchanged unless explicitly planned.
10. Add integration tests for daily/minute merge and quote fallback chain.

### 12.3 Plotly Risk Chart
- `/tmp/dev_manual_assets/risk_distribution.png`

### 12.4 Plotly Migration Impact Chart
- `/tmp/dev_manual_assets/migration_impact.png`

---

## Appendix A: High-Value Snippets (Copy-Paste)

### A1) Core source resolver (`app/utils/workbook_source.py`)
```python
def resolve_financial_data_xlsx(local_candidates: Iterable[str] | None = None) -> Optional[str]:
    del local_candidates
    source_pref = str(os.getenv("FINANCIAL_DATA_SOURCE", DEFAULT_FINANCIAL_DATA_SOURCE)).strip().lower()
    if source_pref not in {"google", "gsheet", "sheet"}:
        logger.info("...forcing Google export")
    sheet_ref = (
        os.getenv("FINANCIAL_DATA_GSHEET_URL")
        or os.getenv("FINANCIAL_DATA_GSHEET_ID")
        or DEFAULT_GOOGLE_SHEET_URL
    )
    sheet_id = extract_google_sheet_id(sheet_ref)
    if not sheet_id:
        return None
    refresh_seconds = int(os.getenv("FINANCIAL_DATA_GSHEET_REFRESH_SECONDS", "60"))
    downloaded = _download_google_sheet_xlsx(sheet_id, refresh_seconds=refresh_seconds)
    if not downloaded or not os.path.exists(downloaded):
        return None
    if not _has_expected_workbook_tabs(downloaded):
        return None
    if not _has_core_financial_coverage(downloaded):
        return None
    return os.path.abspath(downloaded)
```

### A2) Market merge (`app/utils/workbook_market_data.py`)
```python
def load_combined_stock_market_data(...):
    ...
```

### A3) Holders load (`app/utils/workbook_market_data.py`)
```python
def load_holders_sheet(...):
    ...
```

### A4) Overview insight loader (`app/pages/00_Overview.py`)
```python
def _load_overview_insights_sheet(...):
    ...
```

### A5) Transcript highlights renderer (`app/pages/01_Earnings.py`)
```python
def render_transcript_highlights(...):
    ...
```

---

## Appendix B: Open Questions / Unverified Assumptions
1. Optional quote/narrative tabs may vary by workbook revision; alias logic currently mitigates this.
2. Some runtime AI paths still likely depend on historical schema naming assumptions and require branch-level testing.
3. Holders ingestion is stable, but full UX exposure may still be partial depending on page state and filters.
4. Duplicate `pages/` directory at repo root appears legacy; active execution path should remain `app/pages/`.